# M04-P01 — Exploring VAE Latent Space

Objetivo: abrir un VAE pequeño y observar `imagen → μ/logvar → z → reconstrucción`, el espacio latente, interpolaciones y el efecto de `latent_dim`. Todo en CPU.

In [ ]:
import random,time,numpy as np,pandas as pd,matplotlib.pyplot as plt,torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset,DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device("cpu")
print("PyTorch:",torch.__version__); print("Device:",DEVICE)

## 1. Datos

`load_digits` contiene 1.797 imágenes de 8×8. Normalizamos píxeles de 0–16 a 0–1.

In [ ]:
digits=load_digits()
X=digits.data.astype("float32")/16.0; y=digits.target.astype("int64")
images=digits.images.astype("float32")/16.0
print("Ejemplos:",len(X),"X:",X.shape,"Imágenes:",images.shape)
fig,axes=plt.subplots(2,10,figsize=(12,3))
for d in range(10):
    for r,idx in enumerate(np.where(y==d)[0][:2]):
        axes[r,d].imshow(images[idx],cmap="gray",vmin=0,vmax=1); axes[r,d].axis("off")
        if r==0: axes[r,d].set_title(str(d))
plt.tight_layout(); plt.show()

In [ ]:
X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=.2,stratify=y,random_state=SEED)
train_loader=DataLoader(TensorDataset(torch.tensor(X_train),torch.tensor(y_train)),batch_size=128,shuffle=True)
val_loader=DataLoader(TensorDataset(torch.tensor(X_val),torch.tensor(y_val)),batch_size=256)
print("Train:",X_train.shape,y_train.shape); print("Validation:",X_val.shape,y_val.shape)

## 2. VAE

El encoder produce `μ` y `logvar`. Muestreamos `z = μ + exp(0.5·logvar)·ε`, con `ε~N(0,1)`. El decoder reconstruye los 64 píxeles.

In [ ]:
class TinyVAE(nn.Module):
    def __init__(self,latent_dim=2):
        super().__init__(); self.latent_dim=latent_dim
        self.encoder=nn.Sequential(nn.Linear(64,32),nn.ReLU())
        self.mu=nn.Linear(32,latent_dim); self.logvar=nn.Linear(32,latent_dim)
        self.decoder=nn.Sequential(nn.Linear(latent_dim,32),nn.ReLU(),nn.Linear(32,64),nn.Sigmoid())
    def encode(self,x):
        h=self.encoder(x); return self.mu(h),self.logvar(h)
    def reparameterize(self,mu,logvar):
        std=torch.exp(.5*logvar); return mu+torch.randn_like(std)*std
    def decode(self,z): return self.decoder(z)
    def forward(self,x):
        mu,lv=self.encode(x); z=self.reparameterize(mu,lv); return self.decode(z),mu,lv,z
for d in [2,4,8]:
    m=TinyVAE(d)
    print(f"latent_dim={d}: parámetros={sum(p.numel() for p in m.parameters())}")

## 3. Abrir el encoder antes de entrenar

`latent_dim=2` significa que cada imagen se representa mediante una distribución de solo dos dimensiones.

In [ ]:
torch.manual_seed(SEED); demo=TinyVAE(2); xb=torch.tensor(X_train[:4])
with torch.no_grad():
    mu,lv=demo.encode(xb); std=torch.exp(.5*lv); eps=torch.randn_like(std); z=mu+std*eps
display(pd.DataFrame({
    "digit":y_train[:4],
    "mu":[np.round(v.numpy(),4).tolist() for v in mu],
    "logvar":[np.round(v.numpy(),4).tolist() for v in lv],
    "std":[np.round(v.numpy(),4).tolist() for v in std],
    "epsilon":[np.round(v.numpy(),4).tolist() for v in eps],
    "z":[np.round(v.numpy(),4).tolist() for v in z],
}))

## 4. Loss

`VAE loss = reconstruction loss + KL divergence`. Reconstrucción preserva información; KL regulariza el espacio hacia una normal estándar.

In [ ]:
def vae_loss(recon,x,mu,lv):
    rec=F.binary_cross_entropy(recon,x,reduction="sum")/x.size(0)
    kl=-.5*torch.sum(1+lv-mu.pow(2)-lv.exp())/x.size(0)
    return rec+kl,rec,kl

def evaluate(model):
    model.eval(); sums=np.zeros(3); n=0
    with torch.no_grad():
        for xb,_ in val_loader:
            r,mu,lv,_=model(xb); vals=vae_loss(r,xb,mu,lv); b=xb.size(0)
            sums+=np.array([v.item() for v in vals])*b; n+=b
    return sums/n

def train_vae(latent_dim,epochs=60):
    torch.manual_seed(SEED+latent_dim); model=TinyVAE(latent_dim); opt=torch.optim.Adam(model.parameters(),lr=1e-3); hist=[]
    for epoch in range(epochs+1):
        if epoch:
            model.train()
            for xb,_ in train_loader:
                opt.zero_grad(); r,mu,lv,_=model(xb); loss,_,_=vae_loss(r,xb,mu,lv); loss.backward(); opt.step()
        total,rec,kl=evaluate(model); hist.append([epoch,total,rec,kl])
    return model,pd.DataFrame(hist,columns=["epoch","val_total_loss","val_reconstruction_loss","val_kl_loss"])

## 5. Entrenar `latent_dim=2`

In [ ]:
started=time.perf_counter(); vae2,h2=train_vae(2,60)
print(f"Tiempo: {time.perf_counter()-started:.2f} s")
display(h2.iloc[[0,1,5,10,20,40,60]].round(4))
plt.figure(figsize=(8,4))
for col,label in [("val_total_loss","total"),("val_reconstruction_loss","reconstruction"),("val_kl_loss","KL")]:
    plt.plot(h2.epoch,h2[col],label=label)
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("VAE latent_dim=2"); plt.show()

## 6. Reconstrucciones

Fila superior: originales. Fila inferior: reconstrucciones usando `μ` para que la visualización sea estable.

In [ ]:
n=10; xb=torch.tensor(X_val[:n])
with torch.no_grad():
    mu,_=vae2.encode(xb); recon=vae2.decode(mu)
fig,axes=plt.subplots(2,n,figsize=(15,3))
for i in range(n):
    axes[0,i].imshow(xb[i].reshape(8,8),cmap="gray",vmin=0,vmax=1); axes[0,i].axis("off"); axes[0,i].set_title(str(y_val[i]))
    axes[1,i].imshow(recon[i].reshape(8,8),cmap="gray",vmin=0,vmax=1); axes[1,i].axis("off")
axes[0,0].set_ylabel("Original"); axes[1,0].set_ylabel("Recon"); plt.tight_layout(); plt.show()

## 7. Espacio latente 2D

Cada punto es la media `μ` de una imagen de validation.

In [ ]:
def means(model,data):
    with torch.no_grad(): return model.encode(torch.tensor(data))[0].numpy()
z2=means(vae2,X_val)
plt.figure(figsize=(9,7)); sc=plt.scatter(z2[:,0],z2[:,1],c=y_val,cmap="tab10",s=28,alpha=.8)
plt.colorbar(sc,ticks=range(10),label="Dígito"); plt.xlabel("z1"); plt.ylabel("z2"); plt.title("Espacio latente — latent_dim=2"); plt.grid(alpha=.2); plt.show()

## 8. Decodificar el mapa latente

El decoder puede generar imágenes directamente desde coordenadas `z`, sin recibir una imagen original.

In [ ]:
gx=np.linspace(np.percentile(z2[:,0],2),np.percentile(z2[:,0],98),12)
gy=np.linspace(np.percentile(z2[:,1],2),np.percentile(z2[:,1],98),12)
canvas=np.zeros((96,96))
with torch.no_grad():
    for r,zy in enumerate(gy[::-1]):
        for c,zx in enumerate(gx):
            img=vae2.decode(torch.tensor([[zx,zy]],dtype=torch.float32)).reshape(8,8).numpy()
            canvas[r*8:(r+1)*8,c*8:(c+1)*8]=img
plt.figure(figsize=(9,9)); plt.imshow(canvas,cmap="gray",vmin=0,vmax=1); plt.title("Decoder sobre el espacio latente"); plt.axis("off"); plt.show()

## 9. Interpolación

Interpolamos entre la media latente de un `1` y un `7`: `z(t)=(1-t)z₁+t z₂`.

In [ ]:
ia=int(np.where(y_val==1)[0][0]); ib=int(np.where(y_val==7)[0][0])
with torch.no_grad():
    ma,_=vae2.encode(torch.tensor(X_val[ia:ia+1])); mb,_=vae2.encode(torch.tensor(X_val[ib:ib+1]))
    ts=torch.linspace(0,1,11).unsqueeze(1); zs=(1-ts)*ma+ts*mb; imgs=vae2.decode(zs).reshape(-1,8,8)
fig,axes=plt.subplots(1,11,figsize=(14,2))
for i,ax in enumerate(axes):
    ax.imshow(imgs[i],cmap="gray",vmin=0,vmax=1); ax.axis("off"); ax.set_title(f"{float(ts[i]):.1f}")
plt.suptitle("Interpolación latente: 1 → 7"); plt.tight_layout(); plt.show()
print("mu(1):",np.round(ma.numpy(),3)); print("mu(7):",np.round(mb.numpy(),3))

## 10. Comparar `latent_dim=2`, `4` y `8`

Más dimensiones dan más capacidad, pero el VAE equilibra reconstrucción y regularización.

In [ ]:
models={2:vae2}; histories={2:h2}
for d in [4,8]:
    started=time.perf_counter(); models[d],histories[d]=train_vae(d,60); print(f"latent_dim={d}: {time.perf_counter()-started:.2f} s")
rows=[]
for d,m in models.items():
    total,rec,kl=evaluate(m); rows.append({"latent_dim":d,"parameters":sum(p.numel() for p in m.parameters()),"val_total_loss":total,"val_reconstruction_loss":rec,"val_kl_loss":kl})
comparison=pd.DataFrame(rows).sort_values("latent_dim"); display(comparison.round(4))

## 11. Comparar reconstrucciones

In [ ]:
n=8; xb=torch.tensor(X_val[:n]); fig,axes=plt.subplots(4,n,figsize=(12,6))
for i in range(n):
    axes[0,i].imshow(xb[i].reshape(8,8),cmap="gray",vmin=0,vmax=1); axes[0,i].axis("off"); axes[0,i].set_title(str(y_val[i]))
for row,d in enumerate([2,4,8],1):
    with torch.no_grad(): mu,_=models[d].encode(xb); rec=models[d].decode(mu)
    for i in range(n): axes[row,i].imshow(rec[i].reshape(8,8),cmap="gray",vmin=0,vmax=1); axes[row,i].axis("off")
axes[0,0].set_ylabel("Original"); axes[1,0].set_ylabel("latent=2"); axes[2,0].set_ylabel("latent=4"); axes[3,0].set_ylabel("latent=8")
plt.tight_layout(); plt.show()

## 12. Visualizar dimensiones mayores con PCA

PCA 2D es solo una proyección; no es el espacio latente real.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(18,5))
for ax,d in zip(axes,[2,4,8]):
    z=means(models[d],X_val)
    if d==2: zp=z; title="latent_dim=2"
    else:
        pca=PCA(2); zp=pca.fit_transform(z); title=f"latent_dim={d} — PCA var={pca.explained_variance_ratio_.sum():.1%}"
    ax.scatter(zp[:,0],zp[:,1],c=y_val,cmap="tab10",s=20,alpha=.75); ax.set_title(title); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 13. Generar desde `z ~ N(0,I)`

La regularización KL permite muestrear puntos latentes sin partir de una imagen.

In [ ]:
torch.manual_seed(123); fig,axes=plt.subplots(3,10,figsize=(12,4))
for row,d in enumerate([2,4,8]):
    with torch.no_grad(): imgs=models[d].decode(torch.randn(10,d)).reshape(-1,8,8)
    for col in range(10):
        axes[row,col].imshow(imgs[col],cmap="gray",vmin=0,vmax=1); axes[row,col].axis("off")
    axes[row,0].set_ylabel(f"latent={d}")
plt.suptitle("Muestras desde N(0,I)"); plt.tight_layout(); plt.show()

## 14. Qué debes llevarte

Un VAE aprende `encoder → distribución latente` y `decoder → imagen`. La loss combina reconstrucción y KL, favoreciendo un espacio continuo, interpolable y muestreable.

Prueba después `latent_dim=1,16`, otras épocas o interpolar otros dígitos. Pregunta siempre: ¿mejora reconstrucción?, ¿sigue siendo continuo?, ¿qué cambia al generar?

## 15. Reflexión

1. ¿Qué representan μ y logvar? 2. ¿Qué hace el reparameterization trick? 3. ¿Qué significa latent_dim? 4. ¿Por qué necesitamos KL? 5. ¿Por qué una interpolación suave es interesante? 6. ¿Más dimensiones garantizan mejores resultados? 7. ¿Por qué PCA no es el espacio latente real? 8. ¿Cómo puede el decoder generar sin una imagen original?

Siguiente: M04-P02 cambia el enfoque a una GAN: `generador ↔ discriminador`.